In [13]:
import pandas as pd
import numpy as np
from collections import Counter

##### Assumptions
- 1 year of a tender is allocated 1 year of antigen demand (through 1 or many vaccines); scales appropriately
- Intenvory is based on demand for a given year.
- Ratio of inventory to supply is calculated based on end of year totals (after calcs), but only in the programatic sense (check after math, add inventory at end of year for next year)

##### Load and setup demand

In [14]:
# Load the CSV file
demand_path = 'data/real/antigen_demand_80_20_2_scenarios.csv'
data = pd.read_csv(demand_path)
# Create the two dataframes based on the 'prob' column
demand_80 = data[data['prob'] == 0.8]
demand_20 = data[data['prob'] == 0.2]

demand_80 = demand_80.drop(columns=['prob', 'demand_SID'])
demand_20 = demand_20.drop(columns=['prob', 'demand_SID'])
# Expanding the 'demands' column into 10 separate columns
demand_80_expanded = demand_80['demands'].apply(lambda x: pd.Series(eval(x)))
demand_20_expanded = demand_20['demands'].apply(lambda x: pd.Series(eval(x)))

# Renaming the columns to 1-10
demand_80_expanded.columns = range(1, 11)
demand_20_expanded.columns = range(1, 11)

# Concatenating the expanded demands columns back to the original antigen column
demand_80_final = pd.concat([demand_80['antigen'], demand_80_expanded], axis=1)
#added 11th year to capture any left overdemand at the end of year 10.
demand_80_final[11] = 0.1
demand_20_final = pd.concat([demand_20['antigen'], demand_20_expanded], axis=1)
#added 11th year to capture any left overdemand at the end of year 10.
demand_20_final[11] = 0.1

# demand_80_final.head(), demand_20_final.head()


##### Load and setup Starting Points

In [15]:
file_path = 'data/real/Starting_point.xlsx'

# Load the sheets 'F_start', 'I_start', 'S_start' into their own DataFrames
f_start = pd.read_excel(file_path, sheet_name='F_start')
i_start = pd.read_excel(file_path, sheet_name='I_start')
s_start = pd.read_excel(file_path, sheet_name='S_start')


#need to go back throug and re-assign F-start and I-start to new nemes so we dont create extra copies

##### Load pricing data

In [16]:
# Load the Excel file, skipping the first two sheets
money_path = 'data/Vaccine_price_data.xlsx'
sheet_names = pd.ExcelFile(money_path).sheet_names

# Load the remaining sheets into a dictionary of DataFrames
# data = {sheet: pd.read_excel(file_path, sheet_name=sheet) for sheet in sheet_names[2:5]}
price_data = {sheet_names[i]: pd.read_excel(money_path, sheet_name=i).rename(columns=lambda x: "Manufacturer" if x == pd.read_excel(money_path, sheet_name=i).columns[0] else x) for i in range(2, 5)}

# Display the names of the loaded sheets and the first few rows of the first sheet
# sheet_names_loaded = list(price_data.keys())
# first_sheet_preview = data[sheet_names_loaded[0]].head()


##### Load capacity data

In [17]:
# Load the Excel file, only reading the first sheet
capacity_path = 'data/production_capacity_scenarios.xlsx'
sheet_names = pd.ExcelFile(capacity_path).sheet_names

capacity_data = pd.read_excel(capacity_path, sheet_name='base_capacity')
capacity_data

,Manufacturer,1,2,3,4,5,6,7,8,9,10
0,AJ_Vaccines,7711003,7711003,7711003,7711003,7711003,7711003,7711003,7711003,7711003,7711003
1,BB_NCIPD,39223956,39223956,39223956,39223956,39223956,39223956,39223956,39223956,39223956,39223956
2,Bharat_Biotech,61029105,61029105,61029105,61029105,61029105,61029105,61029105,61029105,61029105,61029105
3,Bilthoven,12048153,12048153,12048153,12048153,12048153,12048153,12048153,12048153,12048153,12048153
4,Biological_E,164885690,164885690,164885690,164885690,164885690,164885690,164885690,164885690,164885690,164885690
5,China_National,12812242,12812242,12812242,12812242,12812242,12812242,12812242,12812242,12812242,12812242
6,GSK,226762686,226762686,226762686,226762686,226762686,226762686,226762686,226762686,226762686,226762686
7,Haffkine_Bio,80972855,80972855,80972855,80972855,80972855,80972855,80972855,80972855,80972855,80972855
8,LG_Chem,43702188,43702188,43702188,43702188,43702188,43702188,43702188,43702188,43702188,43702188
9,Merck_Sharp,56991052,56991052,56991052,56991052,56991052,56991052,56991052,56991052,56991052,56991052


##### Initialize stuff

In [18]:
#tender length
delta = 3
vaccine_consumption_percent = 1
years = 10

# Creating an empty DataFrame with the specified structure for calculating ratios
antigens = f_start['Antigen']
columns = ['Antigen',1]

ratio_DF = pd.DataFrame(columns=columns)
ratio_DF['Antigen'] = antigens
ratio_DF[1] = np.zeros(len(antigens))

# create DF to store tender schedule
tender_schedules = f_start.copy()

########################################################
#create DF to store current inventory
inventory_DF = i_start.copy()
##########################################################
#create DF to store missed doses
antigens = f_start['Antigen']
columns = ['Antigen','Missed Doses']

missed_doses = pd.DataFrame(columns=columns)
missed_doses['Antigen'] = antigens
missed_doses['Missed Dosts'] = np.zeros(len(antigens))


In [19]:
def calculate_coverage_and_ratios(inventory_DF, demand_DF, V_a, year):
    """
    Calculates the total coverage of antigens based on vaccine inventory and computes 
    the supply-to-demand ratios for a given year.

    This function takes an inventory DataFrame and a demand DataFrame, along with a 
    dictionary mapping vaccines to antigens, and calculates the total antigen coverage 
    based on the inventory. It then merges this coverage data with the demand data 
    for the specified year to compute the ratio of supply to demand for each antigen.

    Args:
        inventory_DF (pd.DataFrame): A DataFrame containing inventory data with columns 
                                     'Vaccine' and 'Amount', representing the vaccine 
                                     type and the quantity available.
        demand_DF (pd.DataFrame): A DataFrame containing demand data with the first 
                                  column representing antigens and subsequent columns 
                                  representing demand for each year.
        V_a (dict): A dictionary where keys are antigens and values are lists of vaccines 
                    that cover each antigen.
        year (int): The year for which to calculate the supply-to-demand ratio, referring 
                    to the column index in the demand DataFrame.

    Returns:
        tuple: A tuple containing two DataFrames:
            - calculate_ratios_DF (pd.DataFrame): A DataFrame with columns 'antigen', 
              'Total_Coverage', and 'Ratio', representing the supply-to-demand ratio for 
              each antigen.
            - total_coverage_df (pd.DataFrame): A DataFrame with columns 'antigen' and 
              'Total_Coverage', representing the total coverage of each antigen based 
              on the inventory data.
    """

    total_coverage = {antigen: 0 for antigen in V_a.keys()}

    # Iterate across each row in the inventory DataFrame
    for index, row in inventory_DF.iterrows():
        vaccine = row['Vaccine']
        amount = row['Amount']
        
        # For each antigen covered by the vaccine, add the amount to the coverage
        for antigen in V_a.keys():
            if vaccine in V_a[antigen]:
                total_coverage[antigen] += amount

    # Convert the total coverage dictionary to a DataFrame
    total_coverage_df = pd.DataFrame(list(total_coverage.items()), columns=['antigen', 'Total_Coverage'])
    # print(total_coverage_df)

    # Calculate ratio of supply and demand
    calculate_ratios_DF = pd.merge(demand_DF.iloc[:, [0, year]], total_coverage_df, left_on='antigen', right_on='antigen')
    # print(calculate_ratios_DF)
    calculate_ratios_DF['Ratio'] = calculate_ratios_DF.apply(lambda row: 0 if row['Total_Coverage'] == 0 else row['Total_Coverage'] / row[year], axis=1)


    # print(f"Ratio DF:\n{calculate_ratios_DF[['antigen', 'Ratio']]}")

    return calculate_ratios_DF, total_coverage_df

# Example usage:
# result_df = calculate_coverage_and_ratios(inventory_DF_MCV, demand_80_MCV, V_a, year)


##### Initialize antigens/vaccines/prodicers

In [20]:
A = ["Measles", "Mumps", "Rubella"]
V = ["M", "MR", "MMR"]

A_v = {
    "M": ["Measles"],
    "MR": ["Measles", "Rubella"],
    "MMR": ["Measles", "Mumps", "Rubella"]
}

P = ["Biological_E", 
    "GSK","PT_Bio", 
    "Serum_Institute"
]

P_v = {
    "M": ["Serum_Institute", "PT_Bio"],
    "MR": ["Serum_Institute", "Biological_E"],
    "MMR": ["Serum_Institute", "GSK"]
}

P_v = {
    "M": ["Serum_Institute", "PT_Bio"],
    "MR": ["Serum_Institute", "Biological_E"],
    "MMR": ["Serum_Institute", "GSK"]
}


#translate vaccine - antigen, to antigen - vaccine
V_a = {a: [v for v in A_v if a in A_v[v]] for a in A}

V_p = {p: [v for v in P_v if p in P_v[v]] for p in P}

P_a = {a: list(set(p for v in V_a[a] for p in P_v[v])) for a in A}

A_p = {p: [a for a in P_a if p in P_a[a]] for p in P}



In [21]:
def get_manufacturer_vaccine_price(antigen, price_data, Vax_ant_dict, prod_vax_dict, year, find_lowest=False):
    """
    Retrieves a list of manufacturers, vaccines, and prices for a given antigen.

    This function identifies all vaccines that cover a specified antigen by looking 
    up the relevant vaccines in the `Vax_ant_dict` dictionary. It then matches these 
    vaccines to the corresponding manufacturers using the `prod_vax_dict` dictionary 
    and extracts the price information from the provided price data.

    Args:
        antigen (str): The antigen for which to retrieve manufacturer, vaccine, 
                       and price information (e.g., "Measles").
        price_data (dict): A dictionary containing price data with sheet names 
                           as keys (representing vaccines) and DataFrames as values. 
                           Each DataFrame should have a "Manufacturer" column and 
                           price columns.
        Vax_ant_dict (dict): A dictionary mapping antigens to vaccines that include 
                             that antigen.
        prod_vax_dict (dict): A dictionary mapping vaccines to their corresponding 
                              manufacturers.
        find_lowest (bool): If True, returns only the entry with the lowest price. 
                            Default is False.

    Returns:
        list or dict or None: A list of dictionaries, where each dictionary contains 
                              the following keys:
                              - 'Manufacturer': The name of the manufacturer.
                              - 'Vaccine': The name of the vaccine.
                              - 'Price': The price of the vaccine for the given 
                                         manufacturer.
                              If `find_lowest` is True, returns only the dictionary 
                              with the lowest price, or None if no prices are found.
    """
    if antigen not in Vax_ant_dict:
        return f"Antigen pricing not available for {antigen}"


    vaccines = Vax_ant_dict[antigen]
    result = []

    seen_combinations = set()

    for vaccine in vaccines:
        producers = prod_vax_dict[vaccine]
        
        for producer in producers:
            for sheet_name, df in price_data.items():
                if vaccine in sheet_name and producer in df['Manufacturer'].values:
                    # Ensure each manufacturer-vaccine combination is only added once
                    combination = (producer, vaccine)
                    if combination not in seen_combinations:
                        price = df.loc[df['Manufacturer'] == producer, df.columns[year]].values[0]
                        result.append({"Manufacturer": producer, "Vaccine": vaccine, "Price": price})
                        seen_combinations.add(combination)
    
    if not result:
        return None
    
    # Sort the result by vaccine and then by price
    sorted_result = sorted(result, key=lambda x: (x['Vaccine'], x['Price']))
    
    if find_lowest:
        return sorted_result[0]
    
    return sorted_result


In [22]:
get_manufacturer_vaccine_price('Measles', price_data, V_a, P_v, 10)

[{'Manufacturer': 'PT_Bio', 'Vaccine': 'M', 'Price': 0.26664062499999996},
 {'Manufacturer': 'Serum_Institute',
  'Vaccine': 'M',
  'Price': 0.43680859375000003},
 {'Manufacturer': 'Serum_Institute',
  'Vaccine': 'MMR',
  'Price': 2.2771145833333337},
 {'Manufacturer': 'GSK', 'Vaccine': 'MMR', 'Price': 4.47},
 {'Manufacturer': 'Serum_Institute', 'Vaccine': 'MR', 'Price': 0.89083984375},
 {'Manufacturer': 'Biological_E', 'Vaccine': 'MR', 'Price': 0.9200546875}]

# Update this to include a list of all prices/manufacturers from above.

In [23]:
def fulfill_demand(lowest_price_entry, antigen_demand, manufacturer_capacities):
    """
    Fulfills the antigen demand by decrementing manufacturer capacity and calculating the total cost.
    
    Args:
        lowest_price_entry (dict): A dictionary containing the 'Manufacturer', 'Vaccine', 'Price'.
        antigen_demand (int): The total antigen demand to be fulfilled.
        manufacturer_capacities (dict): A dictionary mapping each manufacturer to their production capacity.

        NEED TO ADD:
            - total_price: ability to input put and output current running price
            - antigen demand dynamic updating within function
    
    Returns:
        dict: A dictionary containing the following keys:
            - 'Inventory_Used': The amount of inventory used.
            - 'Demand_Filled': The amount of demand filled.
            - 'Total_Price': The total cost incurred for fulfilling the demand.
            - 'Details': A list of dictionaries with details of how demand was filled, including 'Manufacturer', 
                        'Vaccine', 'Amount_Filled', and 'Price_per_Unit'.
            - 'Message': A message indicating whether the demand was fully met or not.
    """
    
    # Initialize variables
    demand_filled = 0
    total_price = 0
    inventory_used = 0
    details = []

    while antigen_demand > 0:
        manufacturer = lowest_price_entry['Manufacturer']
        vaccine = lowest_price_entry['Vaccine']
        price = lowest_price_entry['Price']

        # Check the manufacturer's capacity
        if manufacturer_capacities[manufacturer] > 0:
            # Determine how much can be filled
            amount_to_fill = min(antigen_demand, manufacturer_capacities[manufacturer])
            
            # Decrement the manufacturer's capacity
            manufacturer_capacities[manufacturer] -= amount_to_fill
            
            # Calculate the total price for this batch
            batch_price = amount_to_fill * price
            
            # Update totals
            demand_filled += amount_to_fill
            total_price += batch_price
            inventory_used += amount_to_fill
            antigen_demand -= amount_to_fill

            # Save the details of this transaction
            details.append({
                'Manufacturer': manufacturer,
                'Vaccine': vaccine,
                'Amount_Filled': amount_to_fill,
                'Price_per_Unit': price
            })
        else:
            # If the manufacturer's capacity is zero, break the loop and return the results
            return {
                'Inventory_Used': inventory_used,
                'Demand_Filled': demand_filled,
                'Total_Price': total_price,
                'Details': details,
                'Message': f"Demand not fulfilled, {antigen_demand} dose {antigen} demand remaining"
            }
    
    # If we exit the loop naturally, the demand was fully met
    return {
        'Inventory_Used': inventory_used,
        'Demand_Filled': demand_filled,
        'Total_Price': total_price,
        'Details': details,
        'Message': "Demand fully fulfilled"
    }

# Example usage:
lowest_price_entry = {
    'Manufacturer': 'Serum_Institute',
    'Vaccine': 'MMR',
    'Price': 2.5,
}

antigen_demand = 500
manufacturer_capacities = {
    'Serum_Institute': 300,
    'GSK': 0,
    'Biological_E': 0,
    'PT_Bio': 0
}

# result = fulfill_demand(lowest_price_entry, antigen_demand, manufacturer_capacities)


## TESTING - Measles Containing Vaccines Only

In [24]:
# Selecting only the rows for 'Measles', 'Mumps', and 'Rubella' in both datasets
demand_80_MCV = demand_80_final[demand_80_final['antigen'].isin(['Measles', 'Mumps', 'Rubella'])]
tender_schedules_MCV = tender_schedules[tender_schedules['Antigen'].isin(['Measles', 'Mumps', 'Rubella'])]
# i_start_MCV = i_start[i_start['Vaccine'].isin(['M', 'MR', 'MMR'])]
missed_doses_MCV = missed_doses[missed_doses['antigen'].isin(['Measles', 'Mumps', 'Rubella'])]
ratio_DF_MCV = pd.DataFrame()
inventory_DF_MCV = inventory_DF[inventory_DF['Vaccine'].isin(['M', 'MR', 'MMR'])]


##### Logic to translate vaccine totals to antigen coverage for later math

In [25]:
#logic to setup least covered antigens:
# Flatten the list of all antigens from all vaccines
all_antigens = [antigen for antigens in A_v.values() for antigen in antigens]
# Count the occurrences of each antigen
antigen_counts = Counter(all_antigens)
# least_covered_antigens = sorted(antigen_counts.keys(), key=lambda x: antigen_counts[x], reverse=False)

In [26]:
antigen_counts

Counter({'Measles': 3, 'Rubella': 2, 'Mumps': 1})

In [27]:
for year in range(1, 4 + 1):  # Iterate through each year - short range for testing  range(1,len(demand_80_MCV.columns)-1)
    print("********************HAPPY NEW YEAR****************************")
    print("******************Reticulating splines...**************************")
    print(f"Year: {year}")

    least_covered_antigens = sorted(antigen_counts.keys(), key=lambda x: antigen_counts[x], reverse=False)
    remaining_inventory = {}
    uncovered_demand = {}
    while least_covered_antigens:  # Iterate through each antigen, find what vaccines cover each antigen, least to greatest, update supply and demand
        print('###############################################################')
        antigen = least_covered_antigens.pop(0)
        print(f"serving antigen {antigen}")
        for vaccine, antigens in A_v.items():  # Iterate through A_v to check which vaccines cover the antigen
            if antigen in antigens and demand_80_MCV.loc[demand_80_MCV.iloc[:, 0] == antigen].iloc[0, year] >0:

                vaccine_inventory_value = inventory_DF_MCV.loc[inventory_DF_MCV.iloc[:, 0] == vaccine].iloc[0, 1]
                # print("iiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiii")
                # print(f"Inventory for {vaccine} for year {year}: ", vaccine_inventory_value)

                antigen_demand_value = demand_80_MCV.loc[demand_80_MCV.iloc[:, 0] == antigen].iloc[0, year]
                # print(f"Demand for {antigen} for year {year}: ", antigen_demand_value)

                difference = vaccine_inventory_value - antigen_demand_value
                if difference >= 0: #
                    remaining_inventory[vaccine] = difference
                    decrement = antigen_demand_value
                else: 
                    remaining_inventory[vaccine] = 0
                    decrement = vaccine_inventory_value
                    uncovered_demand[antigen] = abs(difference)
                    print("------------------------------------------")
                    # print(f"Vaccine:3 {vaccine}, antigen: {antigen}")
                    print(f"uncovered demand for {antigen}: {[antigen]}")
                    #transfer uncovered demand to next year

                # print("^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^")
                inventory_DF_MCV.loc[inventory_DF_MCV.iloc[:, 0] == vaccine, inventory_DF_MCV.columns[1]] = remaining_inventory[vaccine]

                print("Decrementing antigen demands")
                for ant in antigens:
                    if demand_80_MCV.loc[demand_80_MCV.iloc[:, 0] == ant, demand_80_MCV.columns[year]].item() > 0:
                        # print(f"from {ant} demand, reducing demand for year {year} for antigen {ant} by {decrement}")
                        demand_80_MCV.loc[demand_80_MCV.iloc[:, 0] == ant, demand_80_MCV.columns[year]] = demand_80_MCV.loc[demand_80_MCV.iloc[:, 0] == ant].iloc[0, year] - decrement

        print(f"{len(antigen_counts)} antigens entered, only {least_covered_antigens} remain!")

    #update any uncovered demand, to next year. add uncovered demand to dosses_missed dict
    if 'uncovered_demand' in locals(): # Check if the variable exists
        while uncovered_demand:
            top = uncovered_demand.popitem()
            top_antigen = top[0]
            doses_missed = top[1]
            print(f"{doses_missed} doses missed for {top_antigen}")
            demand_80_MCV.loc[demand_80_MCV.iloc[:, 0] == top_antigen, demand_80_MCV.columns[year+1]] += doses_missed
            missed_doses_MCV.loc[missed_doses_MCV.iloc[:, 0] == top_antigen, missed_doses_MCV.columns[1]] += doses_missed
    else:
        print("no uncovered demand this year")
    
    #check ratio for supply/demand.
    #check at end of year for math reasons. if ratio is less than 1, schedule tender, perform search for vaccines, add inventory
    print(f"Checking ratio of supply to demand for antigens for year {year + 1}!")
    print()
    #pulls the current ratio of supply and demand. returns ratio_DF and antigen coverage DF
    ratio_DF_MCV, coverage_df = calculate_coverage_and_ratios(inventory_DF_MCV, demand_80_MCV, V_a, year+1)

    ratio_DF_MCV = ratio_DF_MCV.sort_values(by='antigen', key=lambda x: x.map(antigen_counts), ascending=True)

    for index, row in ratio_DF_MCV.iterrows():
        if row['Ratio'] < 1.0:
            print(f"Ratio: {round(row.loc['Ratio'],2)}")
            print(f"Generating Tender for {row.loc['antigen']}")
            #append F schedule for curreny year +1 to current year +1 + tender_length
            new_row = {'Antigen': row.loc['antigen'], 'Starting': year + 1, 'Ending': year + 4}
            tender_schedules_MCV = pd.concat([tender_schedules_MCV, pd.DataFrame([new_row])], ignore_index=True)
            #inventory search

            price_list = get_manufacturer_vaccine_price(row.loc['antigen'], price_data, V_a, P_v, year + 1)
            # Display the result
            print(f"Lowest Price Information: {price_list}")

        else:
            print(f"Ratio: {round(row.loc['Ratio'],2)}")
            print(f"Supply >= demand for {row.loc['antigen']}")



********************HAPPY NEW YEAR****************************
******************Reticulating splines...**************************
Year: 1
###############################################################
serving antigen Mumps
Decrementing antigen demands
3 antigens entered, only ['Rubella', 'Measles'] remain!
###############################################################
serving antigen Rubella
Decrementing antigen demands
3 antigens entered, only ['Measles'] remain!
###############################################################
serving antigen Measles
Decrementing antigen demands
3 antigens entered, only [] remain!
Checking ratio of supply to demand for antigens for year 2!

Ratio: 1.97
Supply >= demand for Mumps
Ratio: 2.2
Supply >= demand for Rubella
Ratio: 2.1
Supply >= demand for Measles
********************HAPPY NEW YEAR****************************
******************Reticulating splines...**************************
Year: 2
########################################################

In [28]:
missed_doses_MCV

,antigen,Unvaccinated Children
2,Measles,4.580293e+08
3,Mumps,3.203130e+07
8,Rubella,3.407745e+08
